In [ ]:
%matplotlib inline

import os
import random
import numpy as np
import pandas as pd
import yfinance as yf
import gymnasium as gym
from gymnasium import spaces
import optuna
from datetime import datetime
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback

# from finrl.agents import DRLAgent
# from finrl.config import config
from finrl.meta.env_stock_trading.env_stocktrading_np import StockTradingEnv


import random

# import matplotlib
import matplotlib.pyplot as plt

In [ ]:
TRAINING_STEPS = 1_000_000
TENSORBOARD_LOGDIR = "./logs/optuna_forex"
MODEL_DIR = "tmp_models"

In [ ]:
df = yf.download("EURUSD=X", start="2020-01-01", end="2025-10-01", interval="1d")
df = df[['Open', 'High', 'Low', 'Close']].dropna().reset_index(drop=True)

if isinstance(df.columns, pd.MultiIndex):
    df.columns = [col[0].lower() for col in df.columns]  # ne garde que 'open', 'high', etc.

In [ ]:

print(df.head())

In [ ]:
price_array = df[['close']].values
tech_array = np.column_stack([
    df['close'].pct_change().fillna(0),
    df['close'].rolling(5).mean().fillna(method='bfill')
])
turbulence_array = np.zeros(len(df))  # Pas de turbulence réelle ici
if_train = True

config = {
    "price_array": price_array,
    "tech_array": tech_array,
    "turbulence_array": turbulence_array,
    "if_train": if_train
}

In [ ]:
def make_env(config):
    def _init():
        return StockTradingEnv(config=config)
    return _init

env = DummyVecEnv([make_env(config)])

In [ ]:
def optimize_finrl(trial):
    # Hyperparamètres à tester
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 3e-4, log=True)
    n_steps = trial.suggest_categorical("n_steps", [1024, 2048, 4096])
    gamma = trial.suggest_float("gamma", 0.9, 0.9999)
    clip_range = trial.suggest_float("clip_range", 0.1, 0.3)
    ent_coef = trial.suggest_float("ent_coef", 1e-5, 0.01, log=True)

    # Création de l'environnement vectorisé
    train_env = DummyVecEnv([lambda: StockTradingEnv(config=config)])
    
    # Création du modèle
    model = PPO(
        "MlpPolicy",
        train_env,
        learning_rate=learning_rate,
        n_steps=n_steps,
        gamma=gamma,
        clip_range=clip_range,
        ent_coef=ent_coef,
        verbose=0
    )

    # Entraînement (court pour l’optimisation)
    model.learn(total_timesteps=50_000)

    # Évaluation
    mean_reward, _ = evaluate_policy(model, train_env, n_eval_episodes=5)
    return mean_reward

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(optimize_finrl, n_trials=20)  # n_trials = nombre d’expériences

print("Meilleurs hyperparamètres :")
print(study.best_params)
print("Meilleur reward : ", study.best_value)

In [ ]:
class HallucinationCallback(BaseCallback):
    """
    Détecte les comportements incohérents (pnl négatif très élevé, actions extrêmes répétées)
    """
    def __init__(self, env, verbose=0):
        super().__init__(verbose)
        self.env = env
        self.history = []

    def _on_step(self):
        infos = self.locals.get("infos")
        if infos:
            info = infos[0] if isinstance(infos, (list, tuple)) else infos
            if isinstance(info, dict):
                bal = info.get("balance", None)
                pos = info.get("position", None)
                if bal is not None:
                    self.history.append((bal, pos))
                    max_bal = max([h[0] for h in self.history])
                    # On vérifie que bal n'est pas None avant le calcul
                    if max_bal is not None and bal < 0.5 * max_bal:
                        print(f"⚠️ Alerte hallucination : balance {bal:.2f}, position {pos}")
        return True


In [ ]:
best_params = study.best_params
final_env = DummyVecEnv([lambda: StockTradingEnv(config=config)])

final_model = PPO(
    "MlpPolicy",
    final_env,
    learning_rate=best_params["learning_rate"],
    n_steps=best_params["n_steps"],
    gamma=best_params["gamma"],
    clip_range=best_params["clip_range"],
    ent_coef=best_params["ent_coef"],
    verbose=1
)

halluc_cb = HallucinationCallback(final_env)
final_model.learn(total_timesteps=500_000,  callback=halluc_cb)

In [ ]:
config_test = config.copy()
config_test["if_train"] = False
env_test = StockTradingEnv(config=config_test)
actions_list = []

obs = env_test.reset()
if isinstance(obs, tuple):
    obs = obs[0]

total_assets = []
rewards = []

for _ in range(env_test.max_step):
    # Prédiction de l'action à partir de l'observation
    action, _ = final_model.predict(obs, deterministic=True)

    result = env_test.step(action)
    if len(result) == 5:
        obs, reward, terminated, truncated, info = result
        done = terminated or truncated
    else:
        obs, reward, done, info = result

    if isinstance(obs, tuple):
        obs = obs[0]

    total_assets.append(env_test.total_asset)
    rewards.append(reward)
    actions_list.append(action)
    
    if done:
        break

print("✅ Test terminé")


In [ ]:
# --- Conversion des actions en points d'achat/vente ---
actions_list = np.array(actions_list)
buy_points = np.where(actions_list > 0)[0]
sell_points = np.where(actions_list < 0)[0]

# --- Tracé du portefeuille ---
plt.figure(figsize=(12, 6))
plt.plot(total_assets, label="Valeur du portefeuille", color="blue")

plt.scatter(buy_points, np.array(total_assets)[buy_points], color="green", label="Achat", marker="^", alpha=0.8)
plt.scatter(sell_points, np.array(total_assets)[sell_points], color="red", label="Vente", marker="v", alpha=0.8)

plt.title("📈 Performance du modèle FinRL sur EUR/USD")
plt.xlabel("Étapes (jours)")
plt.ylabel("Valeur du portefeuille (€)")
plt.legend()
plt.grid(True)
plt.show()

print(f"💰 Solde initial : {env_test.initial_capital:.2f}")
print(f"💰 Solde final : {env_test.total_asset:.2f}")
print(f"📈 Gain total : {(env_test.total_asset/env_test.initial_capital - 1)*100:.2f}%")

In [ ]:
# Dossier de sauvegarde
models_dir = "models"
os.makedirs(models_dir, exist_ok=True)  # crée le dossier s'il n'existe pas

# Chemin complet du modèle
model_path = os.path.join(models_dir, "ppo_forex_final.zip")

# Sauvegarde
final_model.save(model_path)
print(f"💾 Modèle sauvegardé : {model_path}")